In [4]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

一个完整的Agent至少要包含两个关键的部分：
- **模型**：是Agent的大脑，负责推理、分析，规划任务步骤
- **工具**：是Agent的手脚，负责执行任务，与外界交互

因此，定义带有工具的Agent的基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

# 1.自定义工具

所谓的**工具（Tool）**，本质就是一个可调用的**函数**，但是这个函数不是我们自己去调用，而是给模型调用。因此除了定义函数外，我们还需要清晰描述这个工具，让模型知道这个工具如何使用。包括下列信息：
- 工具名
- 工具的作用
- 工具需要的参数


## 1.1.基于tool描述工具
在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用：


In [3]:
from langchain_core.tools import tool

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

## 1.2.使用函数名和文档注释描述工具

如果不@tool装饰器没有定义工具名和作用描述，此时：
- 工具名：默认就是函数名
- 工具所需的参数：默认就是函数的参数列表
- 工具作用的描述：默认就是函数的文档注释

In [4]:
from langchain_core.tools import tool
# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [5]:
# 定义一个查询天气的tool
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

## 1.3.定义Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表。


In [6]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal


# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference, default is celsius."
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


工具调用方式与普通函数调用方式一致。


In [9]:
square_root.invoke({"x": 9})

3.0

In [11]:
get_weather.invoke({"location": "广州", "include_forecast": True})

'Current weather in 广州: 22 degrees C\nNext 5 days: Sunny'

## 测试

In [10]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 创建智能体
llm = ChatOpenAI(
    model="qwen3.7-max-2026-06-08",
    base_url=base_url,
    api_key=api_key,
    extra_body={"enable_thinking": False},  # 关闭思考模式
)

agent = create_agent(
    model=llm,
    tools=[square_root, get_weather],
    system_prompt="你可以使用工具回答用户问题，调用工具时尽量使用默认参数，除非用户特别指定。"
)

NameError: name 'square_root' is not defined

In [13]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="广州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


Current weather in 广州: 22 degrees C
Next 5 days: Sunny广州当前气温为22摄氏度，接下来5天的天气预报为晴天。

In [15]:
response = agent.invoke(
    {"messages": [HumanMessage(content="99999和11111的平方根是多少?")]},
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

99999和11111的平方根是多少?
None
================================== Ai Message ==================================
Tool Calls:
  square_root (call_131bb792361544ae8fc476d0)
 Call ID: call_131bb792361544ae8fc476d0
  Args:
    x: 99999
  square_root (call_8f2e0104f3b34f5d8e60ca3a)
 Call ID: call_8f2e0104f3b34f5d8e60ca3a
  Args:
    x: 11111
None
================================= Tool Message =================================
Name: square_root

316.226184874055
None
================================= Tool Message =================================
Name: square_root

105.40872829135166
None
================================== Ai Message ==================================

99999的平方根约为316.23，11111的平方根约为105.41。
None


完整流程如图：
<img src="./resources/agent-flow2.png">

# 2.预定义Tool

LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily：就是一个用来做web搜索的工具

## 2.1.基本用法
它的使用步骤是这样的：
- 注册账号，创建API_KEY
- 配置环境变量: TAVILY_API_KEY
- 安装依赖：`uv add langchain-tavily`


In [5]:
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=2,
    topic="general", # general, news, finance
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [7]:
search_tool.invoke("鸡你太美是什么梗？")

{'query': '鸡你太美是什么梗？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://m.ali213.net/news/gl2401/1298525.html',
   'title': '鸡你太美什么梗 鸡你太美梗的由来_游侠手游',
   'content': '当前位置：全部攻略 > 软件综合 > 梗百科 > 正文. # 鸡你太美什么梗. 鸡你太美什么梗？有很多老梗一直都在，而且不失流行，但是感觉这些梗也不太好，这个鸡你太美不知道是怎么流行起来的，下面就跟着小编一起来了解一下鸡你太美的梗吧。. "鸡你太美"是一个网络流行语，源自当红流量明星蔡徐坤在打篮球跳舞时所唱的歌词"只因你太美"的谐音。网友们用这个词语来调侃蔡徐坤打球技巧差、唱歌水平低，常常出现在B站的鬼畜视频和弹幕中。. 这个词语的起源可以追溯到蔡徐坤成为NBA形象大使之后，他的一段篮球舞在互联网上广为流传。这个舞蹈的背景音乐(BGM)是"只因你太美"，但蔡徐坤唱得太快了，导致歌词听起来像"鸡你太美"。. 因此，这个词语就被网友们用来形容某个人打球差、唱歌差的情况，成为了一个具有调侃意味的网络流行语。在B站的鬼畜视频和弹幕中，常常可以看到这个词语的出现，成为了一种常见的调侃方式。. 关于游侠 | 广告合作 | 人才招聘 | 投诉建议 | 联系我们 | 网站地图 | 版权保护投诉指引 | app上传. CopyRight © 1999-2023 ALi213.Net All Right Reserved | 苏ICP备2023007791号.',
   'score': 0.8875269,
   'raw_content': None},
  {'url': 'https://baike.baidu.com/item/%E9%B8%A1%E4%BD%A0%E5%A4%AA%E7%BE%8E/53592938',
   'title': '鸡你太美（网络流行语）_百度百科',
   'content': '# 鸡你太美. 鸡你太美在一定程度上是当代网友逆反心理的真实写照，其次，这个梗能够风靡不衰是其背后所携带的巨大流量。鸡你太美不断被创新，不断衍生出新的形式，吸

In [12]:
# 创建智能体，使用预定义工具tavily
llm = ChatOpenAI(
    model="qwen3.7-max-2026-06-08",
    base_url=base_url,
    api_key=api_key,
    extra_body={"enable_thinking": False},  # 关闭思考模式
)

agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。"
)

In [13]:
response = agent.invoke(
    {"messages": [HumanMessage(content="鸡你太美是什么梗？")]},
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

鸡你太美是什么梗？
================================== Ai Message ==================================

“鸡你太美”是中国互联网上非常知名的一个**空耳梗**（指将一种语言的声音故意或无意听成另一种语言的谐音），源自艺人蔡徐坤的一段表演视频。

以下是该梗的详细解析：

### 1. 梗的出处
*   **来源：** 2018年偶像男团竞演养成类真人秀《偶像练习生》。
*   **原曲：** SWIN-S组合的歌曲《只因你太美》。
*   **名场面：** 在节目中，蔡徐坤进行个人才艺展示时，演唱了这首歌曲并配合篮球舞蹈表演。由于演唱时语速较快且带有独特的转音，歌词“**只因你太美**”被大量网友空耳听成了“**鸡你太美**”。

### 2. 爆火原因
这个梗之所以能从单纯的“听错”演变成全网狂欢，主要有以下几个因素：
*   **视听反差：** 视频中精致的妆容、激烈的篮球动作与背景音乐形成了强烈的视觉冲击，加上空耳歌词的滑稽感，产生了意想不到的喜剧效果。
*   **B站二创推动：** 2019年初，Bilibili等平台上涌现出大量基于该片段制作的鬼畜视频、恶搞剪辑和表情包。这些二次创作极大地加速了梗的传播，使其脱离了原本的粉丝圈层，成为大众网络迷因。
*   **律师函事件：** 蔡徐坤工作室曾向B站发送律师函要求下架相关视频，这一举动反而引发了网友的逆反心理，导致相关二创视频呈爆发式增长，“鸡你太美”也因此彻底出圈。

### 3. 衍生文化与现状
*   **符号化：** “鸡”、“篮球”、“背带裤”、“中分头”成为了与该梗绑定的核心视觉符号。
*   **语义泛化：** 如今该梗已逐渐脱离了对艺人本身的攻击性，更多时候被用作一种**无意义的语气助词**、**搞笑素材**或**社交暗号**。当人们提到“鸡你太美”时，往往只是在玩梗或表达一种戏谑的情绪，而非针对歌曲或歌手本人。
*   **官方态度变化：** 随着时间推移，当事人及公众对该梗的态度也趋于平和，它已成为中国互联网亚文化中一个具有代表性的历史印记。



## 2.2.优化

目前的搜索智能体存在两个问题：
- 官方默认的tavily工具过于复杂
- 结果中不包含网页数据源，可信度低

解决思路：
- 自定义tavily工具
- 结构化输出

### 自定义tavily工具

LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和Token消耗。因此，对于简单的业务，我们建议大家利用tavily自定义工具。


In [17]:
from langchain_core.tools import tool

# 先使用官方的客户端做初始化
tavily = TavilySearch(
    max_results=5,
    topic="general"
)

# 然后自己封装为tool
@tool
def web_search(query: str):
    """Search the web for information"""
    return tavily.invoke(query)

### 定义结构化输出实体


In [18]:
from pydantic import BaseModel, Field

# Agent回答内容引用的网页信息
class Reference(BaseModel):
    title: str = Field(description="The title of the web page cited in the answer")
    url: str = Field(description="The url of the web page cited in the answer")

# Agent的回答内容
class AnswerInfo (BaseModel):
    answer: str = Field(description="The final answer for user")
    reference: list[Reference] = Field(description="The web pages cited in the answer")

In [19]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)

In [20]:
# 调用agent
response = agent.invoke(
    {"messages": [HumanMessage(content="鸡你太美是什么梗？")]},
)

# 获取结构化输出
print(response['structured_response'])

answer='“鸡你太美”是一个源自艺人蔡徐坤的网络流行梗，其具体含义和起源如下：\n\n1.  **词源出处**：该梗出自男团选秀节目《偶像练习生》中蔡徐坤的一段篮球舞表演。背景音乐是其所在组合SWIN的歌曲《只因你太美》，但由于演唱时语速过快、发音模糊，第一句歌词“只因你太美”被网友空耳（听错）成了“鸡你太美”。\n2.  **爆火原因**：这段结合了篮球运球与舞蹈的表演视频在网络上广泛传播后，因动作风格和妆造引发争议，被大量网友作为鬼畜、表情包等二次创作的素材，“鸡你太美”也随之成为调侃蔡徐坤及其相关表演的代名词。\n3.  **后续影响**：该梗后来逐渐演变为一种带有讽刺或戏谑意味的网络用语，甚至衍生出大量二创作品。但因其部分内容存在低俗化倾向，曾被人民网等媒体点名批评为“网络烂梗”，提醒需警惕其对青少年的不良影响。' reference=[Reference(title='鸡你太美什么梗 - 游侠手游', url='https://m.ali213.net/news/gl2401/1298525.html'), Reference(title='“鸡你太美”蔡徐坤-钛媒体官方网站', url='https://www.tmtpost.com/3864316.html'), Reference(title='鸡你太美 - 萌娘百科', url='https://zh.moegirl.org.cn/%E9%B8%A1%E4%BD%A0%E5%A4%AA%E7%BE%8E'), Reference(title='人民网评「鸡你太美」是低俗烂梗，青少年爱说的烂梗越来越多', url='https://www.zhihu.com/question/588032253/answer/2973797630')]
